In [ ]:
import numpy as np
from scipy.signal import welch
import matplotlib.pyplot as plt
import seaborn as sns
import sys
if not '..' in sys.path:
    sys.path.append('..')
from pfcommon import OU, OU_2, OU_signal

In [ ]:
tau_ref = 20e-3
stddev_ref = 0.01 * 322
DC_gain = 2 * tau_ref * stddev_ref ** 2
print('Reference τ: {:g} s'.format(tau_ref))
print('Reference σ: {:g}'.format(stddev_ref))
print('DC gain: {:g}'.format(DC_gain))

In [ ]:
tend = 5000
dt = 1e-3
mean, tau = 0, 200e-3
stddev = np.sqrt(DC_gain / (2 * tau))
print('Reference τ: {:g} s'.format(tau))
print('Reference σ: {:g}'.format(stddev))
print('DC gain: {:g}'.format(2 * tau * stddev ** 2))
μ, c, α = mean, stddev * np.sqrt(2 / tau), 1 / tau
# μ, c, α = 0, 1, 10 * 2 * np.pi
# mean, stddev, tau = μ, np.sqrt(c**2 / (2 * α)), 1 / α
time = np.r_[0 : tend+dt/2 : dt]
N = time.size
cutoff = α / (2*np.pi)
seed = 100
x = OU(dt, mean, stddev, tau, N, seed)
x_2 = OU_2(dt, α, μ, c, N, seed)
x_ref = OU(dt, mean, stddev_ref, tau_ref, N, seed)
idx = time > 10 * tau

In [ ]:
srate = 1 / dt
Δf = cutoff / 1000
Δf = 5e-3
nperseg = int(srate / Δf)
onesided = True
kwargs = {
    'fs': srate,
    'window': 'hamming',
    'nperseg': nperseg,
    'noverlap': nperseg // 2,
    'return_onesided': onesided,
    'scaling': 'density',
    'detrend': None
}

freq, P = welch(x[idx] - x[idx].mean(), **kwargs)
_, P_2 = welch(x_2[idx] - x_2[idx].mean(), **kwargs)
_, P_ref = welch(x_ref[idx] - x_ref[idx].mean(), **kwargs)
if onesided:
    P /= 2
    P_2 /= 2
    P_ref /= 2
else:
    Nf = freq.size
    freq = freq[:Nf//2]
    P = P[:Nf//2]
    P_2 = P_2[:Nf//2]
    P_ref = P_ref[:Nf//2]
P_theor = (c / α) ** 2 / (1 + (2 * np.pi * freq / α) ** 2)
dc_gain = 2 * tau * stddev ** 2
assert abs(P_theor[0] - dc_gain) < 1e-8
print('DC gain: {:g} ({:.2f} dB)'.format(dc_gain, 10 * np.log10(dc_gain)))

The variance of a process is the integral of its PSD over all frequencies:

In [ ]:
print('Integral of theoretical PSD: {:g}'.format(np.trapezoid(P_theor, freq)))
print('  OU >> variance: {:g}, integral: {:g}'.format(x[idx].var(), np.trapezoid(P, freq) * (1 + onesided)))
print('OU_2 >> variance: {:g}, integral: {:g}'.format(x_2[idx].var(), np.trapezoid(P_2, freq) * (1 + onesided)))

In [ ]:
P_db = 10 * np.log10(P)
P_2_db = 10 * np.log10(P_2)
P_ref_db = 10 * np.log10(P_ref)
P_theor_db = 10 * np.log10(P_theor)

In [ ]:
fig,ax = plt.subplots(1, 1, figsize=(5,3))
ax.plot(freq, P_db, 'k', lw=0.75, label='OU (τ={:.0f} ms)'.format(tau * 1e3))
ax.plot(freq, P_2_db, 'tab:green', lw=0.75, alpha=0.75, label='OU_2 (τ={:.0f} ms)'.format(tau * 1e3))
ax.plot(freq, P_ref_db, 'tab:blue', lw=0.75, alpha=0.75, label='OU (τ={:.0f} ms)'.format(tau_ref * 1e3))
ax.plot(freq, P_theor_db, 'm', lw=2, label='Theory')
yl = ax.get_ylim()
ax.vlines(cutoff, yl[0], yl[1], ls='--', color='tab:red', lw=1.5, label='1/τ')
ax.hlines(10 * np.log10(dc_gain), freq[0], freq[-1], ls='--', color=[.4, .8, .8], lw=1.5, label='DC gain')
ax.legend(loc='lower left', fontsize=8, frameon=False)
ax.grid(which='major', axis='y', lw=0.5, ls=':', color=[.6,.6,.6])
ax.set_xscale('log')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('PSD')
sns.despine()
fig.tight_layout()